# End-sem 01. Data lake (PySpark + Parquet) and player entity resolution
Sources: FBref 2017-18 to 2023-24 (the mid-sem file) plus 2024-25 (`emrey3lmaz`, scraped before FBref removed its advanced columns), Understat team style, Transfermarkt (`davidcariboo/player-scores`).
Everything is written as Parquet partitioned by `season` and `league`. FBref players are then linked to Transfermarkt ids.

In [1]:
import sys, os; sys.path.insert(0, 'src')
import pandas as pd, numpy as np
from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import DoubleType, StringType
from common import *
import style as ST
sp = spark(); RT = {}
TM = 'data/transfermarkt'
rd = lambda name: sp.read.csv(f'{TM}/{name}.csv', header=True, inferSchema=True, multiLine=True, escape='"')
lg = F.create_map(*[F.lit(x) for kv in TM_LEAGUE.items() for x in kv])
season_of = lambda d: F.when(F.month(d) >= 7, F.year(d)).otherwise(F.year(d) - 1)
def write(df, name, parts=('season', 'league')):
    df.write.mode('overwrite').partitionBy(*parts).parquet(f'lake/{name}'); n = sp.read.parquet(f'lake/{name}').count(); print(f'lake/{name}: {n:,} rows'); return n

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 01:48:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Write every source as Parquet

In [2]:
rows = {}
with timed('write lake', RT):
    RAWP, src_agree, n_chk = ST.load_raw()      # 2017-18 to 2023-24 from the mid-sem file, 2024-25 from the emrey3lmaz scrape
    raw = sp.createDataFrame(RAWP).withColumn('season', F.col('yr')).withColumn('league', F.col('comp')); rows['fbref_raw'] = write(raw, 'fbref_raw')
    clubs = rd('clubs').select('club_id', F.col('name').alias('club_name'), F.coalesce(lg[F.col('domestic_competition_id')], F.lit('other')).alias('club_league'))
    tr = (rd('transfers').withColumn('transfer_date', F.to_date('transfer_date')).withColumn('season', (F.lit(2000) + F.substring('transfer_season', 1, 2).cast('int')))
          .filter(F.col('season').between(2010, 2026))
          .join(F.broadcast(clubs.select(F.col('club_id').alias('to_club_id'), F.col('club_league').alias('league'))), 'to_club_id', 'left').fillna({'league': 'other'}))
    rows['tm_transfers'] = write(tr, 'tm_transfers')
    val = (rd('player_valuations').withColumn('date', F.to_date('date')).withColumn('season', season_of('date')).filter(F.col('season') >= 2010)
           .withColumn('league', F.coalesce(lg[F.col('player_club_domestic_competition_id')], F.lit('other'))))
    rows['tm_valuations'] = write(val, 'tm_valuations')
    pl = (rd('players').withColumn('birth_year', F.year(F.to_date('date_of_birth'))).withColumn('season', F.col('last_season'))
          .withColumn('league', F.coalesce(lg[F.col('current_club_domestic_competition_id')], F.lit('other'))).drop('image_url', 'url', 'agent_name'))
    rows['tm_players'] = write(pl, 'tm_players')
    # player x club x season league minutes (needed for blocking and for "did the starter still play next season")
    app = (rd('appearances').filter(F.col('competition_id').isin(list(TM_LEAGUE))).withColumn('season', season_of(F.to_date('date')))
           .groupBy('player_id', F.col('player_club_id').alias('club_id'), 'season', lg[F.col('competition_id')].alias('league'))
           .agg(F.sum('minutes_played').alias('minutes'), F.count('*').alias('games')))
    rows['tm_club_minutes'] = write(app, 'tm_club_minutes')
    # final league tables from match results (covers 2024-25 and 2025-26, which the Understat table does not)
    g = rd('games').filter(F.col('competition_id').isin(list(TM_LEAGUE))).select('season', lg[F.col('competition_id')].alias('league'), 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals')
    side = lambda me, op, gf, ga: g.select('season', 'league', F.col(me).alias('club_id'), F.col(gf).alias('gf'), F.col(ga).alias('ga'))
    tab = (side('home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals').unionByName(side('away_club_id', 'home_club_id', 'away_club_goals', 'home_club_goals'))
           .groupBy('season', 'league', 'club_id').agg(F.count('*').alias('matches'), F.sum(F.when(F.col('gf') > F.col('ga'), 3).when(F.col('gf') == F.col('ga'), 1).otherwise(0)).alias('pts'), F.sum(F.col('gf') - F.col('ga')).alias('gd'))
           .withColumn('tm_pos', F.row_number().over(W.partitionBy('season', 'league').orderBy(F.desc(F.col('pts') / F.col('matches')), F.desc('gd')))))
    rows['tm_tables'] = write(tab, 'tm_tables')
sp.read.parquet('lake/fbref').filter("season = 2023 and league = 'Ligue 1'").select('player', 'squad', 'min_num').explain()

lake/fbref_raw: 21,097 rows


lake/tm_transfers: 168,248 rows


lake/tm_valuations: 615,190 rows


lake/tm_players: 50,149 rows


lake/tm_club_minutes: 38,935 rows


lake/tm_tables: 1,366 rows
[write lake] 19.8 s
== Physical Plan ==
*(1) Project [player#484, squad#485, min_num#491L]
+- *(1) ColumnarToRow
   +- FileScan parquet [player#484,squad#485,min_num#491L,season#555,league#556] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/srioo/Desktop/Projects/root/BigP/bigdata/lake/fbref], PartitionFilters: [isnotnull(season#555), isnotnull(league#556), (season#555 = 2023), (league#556 = Ligue 1)], PushedFilters: [], ReadSchema: struct<player:string,squad:string,min_num:bigint>




## 2. Entity resolution: FBref player-season rows to Transfermarkt player ids
**Pass 1 (learn the club alias map).** Join on (season, league, birth year, exact name key) and take, for every FBref club, the Transfermarkt club most of its exactly-matched players belong to.
**Pass 2 (the match).** Block on (season, club). Inside a block compare name keys with Spark `levenshtein` and a Jaro-Winkler pandas UDF, add birth year and position-group agreement, keep mutual best pairs.

In [3]:
from rapidfuzz.distance import JaroWinkler
pkey_u = F.udf(pkey, StringType())
@F.pandas_udf(DoubleType())
def jw(a: pd.Series, b: pd.Series) -> pd.Series:
    return pd.Series([JaroWinkler.similarity(x, y) for x, y in zip(a, b)])
@F.pandas_udf(DoubleType())
def subset(a: pd.Series, b: pd.Series) -> pd.Series:   # "Rodri" vs "Rodrigo Hernandez Rodri": every token of the shorter name appears in the longer one
    return pd.Series([float(bool(x) and bool(y) and (set(x.split()) <= set(y.split()) or set(y.split()) <= set(x.split()))) for x, y in zip(a, b)])

with timed('entity resolution', RT):
    raw = sp.read.parquet('lake/fbref_raw').filter(~F.upper('pos').startswith('GK'))
    fbp = (raw.select('player', 'squad', F.col('league').alias('comp'), F.col('season').alias('yr'), F.col('born').cast('int').alias('born'), F.upper(F.substring('pos', 1, 2)).alias('pos2'), 'min_num')
           .withColumn('k_fb', pkey_u('player')).dropDuplicates(['player', 'squad', 'comp', 'yr']).cache())
    tmp = (sp.read.parquet('lake/tm_club_minutes').join(sp.read.parquet('lake/tm_players').select('player_id', 'name', 'birth_year', 'sub_position', 'position'), 'player_id')
           .filter(F.col('position') != 'Goalkeeper').withColumnRenamed('season', 'yr').withColumnRenamed('league', 'comp').withColumn('k_tm', pkey_u('name')).cache())
    # pass 1: club alias map by majority vote of exact matches
    votes = (fbp.join(tmp, (fbp.yr == tmp.yr) & (fbp.comp == tmp.comp) & (fbp.born == tmp.birth_year) & (fbp.k_fb == tmp.k_tm)).groupBy(fbp.comp, 'squad', 'club_id').count()
             .withColumn('r', F.row_number().over(W.partitionBy('comp', 'squad').orderBy(F.desc('count')))).withColumn('share', F.col('count') / F.sum('count').over(W.partitionBy('comp', 'squad'))).filter('r = 1').drop('r'))
    club_x = votes.join(F.broadcast(clubs.select('club_id', 'club_name')), 'club_id').toPandas()
    all_sq = fbp.select('comp', 'squad').distinct().toPandas()
    assert len(club_x) == len(all_sq) and club_x['club_id'].is_unique, 'club map is not one-to-one / complete'
    sp.createDataFrame(club_x).write.mode('overwrite').parquet('lake/club_xref')
    # pass 2: block on (season, club)
    cx = F.broadcast(sp.createDataFrame(club_x[['comp', 'squad', 'club_id']]))
    cand = (fbp.join(cx, ['comp', 'squad']).join(tmp.drop('comp'), ['yr', 'club_id'])
            .withColumn('lev', 1 - F.levenshtein('k_fb', 'k_tm') / F.greatest(F.length('k_fb'), F.length('k_tm'), F.lit(1)))
            .withColumn('name_sim', F.greatest('lev', jw('k_fb', 'k_tm'), F.lit(0.9) * subset('k_fb', 'k_tm')))
            .withColumn('by', F.when(F.col('born') == F.col('birth_year'), 1.0).when(F.abs(F.col('born') - F.col('birth_year')) == 1, 0.5).otherwise(0.0))
            .withColumn('pos_ok', (((F.col('pos2') == 'DF') & (F.col('position') == 'Defender')) | ((F.col('pos2') == 'MF') & (F.col('position') == 'Midfield')) | ((F.col('pos2') == 'FW') & (F.col('position') == 'Attack'))).cast('double'))
            .withColumn('confidence', F.round(0.6 * F.col('name_sim') + 0.3 * F.col('by') + 0.1 * F.col('pos_ok'), 3))
            .filter((F.col('name_sim') >= 0.80) & (F.col('by') > 0) | (F.col('name_sim') >= 0.95)))
    best = (cand.withColumn('r1', F.row_number().over(W.partitionBy('player', 'squad', 'yr').orderBy(F.desc('confidence'), F.desc('minutes'))))
                .withColumn('r2', F.row_number().over(W.partitionBy('player_id', 'club_id', 'yr').orderBy(F.desc('confidence'), F.desc('min_num')))).filter('r1 = 1 and r2 = 1'))
    fine = F.create_map(*[F.lit(x) for kv in FINE_POS.items() for x in kv])
    best = best.cache(); cols = ['player', 'squad', 'comp', 'yr', 'born', 'player_id', F.col('name').alias('tm_name'), 'club_id', 'birth_year', 'sub_position', fine[F.col('sub_position')].alias('fine_pos'), 'name_sim', 'by', 'pos_ok', 'confidence', F.col('minutes').alias('tm_minutes'), 'min_num']
    # pass 3: names that do not resemble each other (nicknames, e.g. FBref "Jonny Castro" = Transfermarkt "Jonny Otto"). Same club and season, same birth year and position group, league minutes within 10%, and the pair must be unique on both sides.
    left_fb = fbp.join(cx, ['comp', 'squad']).join(best.select('player', 'squad', 'yr'), ['player', 'squad', 'yr'], 'left_anti').filter('min_num >= 450')
    left_tm = tmp.drop('comp').join(best.select('player_id', 'club_id', 'yr'), ['player_id', 'club_id', 'yr'], 'left_anti')
    posm = ((F.col('pos2') == 'DF') & (F.col('position') == 'Defender')) | ((F.col('pos2') == 'MF') & (F.col('position') == 'Midfield')) | ((F.col('pos2') == 'FW') & (F.col('position') == 'Attack'))
    p3 = (left_fb.join(left_tm, ['yr', 'club_id']).filter((F.col('born') == F.col('birth_year')) & posm & (F.abs(F.col('min_num') - F.col('minutes')) <= 0.1 * F.greatest('min_num', 'minutes') + 45))
          .withColumn('n1', F.count('*').over(W.partitionBy('player', 'squad', 'yr'))).withColumn('n2', F.count('*').over(W.partitionBy('player_id', 'club_id', 'yr'))).filter('n1 = 1 and n2 = 1')
          .withColumn('name_sim', 1 - F.levenshtein('k_fb', 'k_tm') / F.greatest(F.length('k_fb'), F.length('k_tm'), F.lit(1))).withColumn('by', F.lit(1.0)).withColumn('pos_ok', F.lit(1.0)).withColumn('confidence', F.lit(0.5)))
    xref = best.select(*cols, F.lit('name').alias('method')).unionByName(p3.select(*cols, F.lit('birth year + minutes').alias('method')))
    xref.withColumn('season', F.col('yr')).withColumn('league', F.col('comp')).write.mode('overwrite').partitionBy('season', 'league').parquet('lake/player_xref')
cand.select('player', 'tm_name' if False else 'name', 'confidence').explain()
X = sp.read.parquet('lake/player_xref').toPandas(); FB = fbp.toPandas()
print(f'{len(club_x)} FBref clubs mapped one-to-one to Transfermarkt clubs (lowest vote share {club_x["share"].min():.2f}); {len(X):,} of {len(FB):,} outfield player-club-seasons linked')

[entity resolution] 8.5 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [player#558, name#666, round((((greatest((1.0 - (cast(levenshtein(k_fb#591, k_tm#694, None) as double) / cast(greatest(length(k_fb#591), length(k_tm#694), 1) as double))), pythonUDF0#4710, (0.9 * pythonUDF1#4711)) * 0.6) + CASE WHEN (born#588 = birth_year#686) THEN 0.3 WHEN (abs((born#588 - birth_year#686)) = 1) THEN 0.15 ELSE 0.0 END) + (cast(((((pos2#589 = DF) AND (position#675 = Defender)) OR ((pos2#589 = MF) AND (position#675 = Midfield))) OR ((pos2#589 = FW) AND (position#675 = Attack))) as double) * 0.1)), 3) AS confidence#1488]
   +- ArrowEvalPython [jw(k_fb#591, k_tm#694)#1483, subset(k_fb#591, k_tm#694)#1484], [pythonUDF0#4710, pythonUDF1#4711], 200
      +- Project [player#558, born#588, pos2#589, k_fb#591, name#666, birth_year#686, position#675, k_tm#694]
         +- Filter (((greatest((1.0 - (cast(levenshtein(k_fb#591, k_tm#694, None) as double) / cast(greatest(length(k_fb#591), len

149 FBref clubs mapped one-to-one to Transfermarkt clubs (lowest vote share 0.82); 19,166 of 19,507 outfield player-club-seasons linked


## 3. Player-season feature table with a walk-forward style model
One row per player-season with 900+ minutes (the club where he played most). Team style comes from Understat. For every season y the style elasticities and league offsets are refitted on seasons up to y, so the style-neutral numbers of a season never use a later one. League positions come from Transfermarkt match results, with the mid-sem table kept where it exists.

In [4]:
with timed('walk-forward style model', RT):
    D = RAWP[(~RAWP['pos'].astype(str).str.upper().str.startswith('GK')) & (RAWP['min_num'] >= 900)].copy(); D['pos_group'] = D['pos'].astype(str).str.upper().str[:2]
    D = D[D['pos_group'].isin(ST.POS3)].sort_values('min_num', ascending=False).drop_duplicates(['player', 'yr'])
    ctx, miss = ST.team_style(RAWP[['comp', 'squad']].drop_duplicates()); assert not miss, f'Understat clubs without an FBref name: {miss}'
    D = D.merge(ctx[['comp', 'squad', 'yr', 'x_dom', 'x_press', 'x_str', 'matches']], on=['comp', 'squad', 'yr'], how='left'); assert D['x_dom'].notna().all(), 'club-season without Understat style'
    FE, ELW, OFW = ST.walk_forward_features(D, range(2017, 2025))
    tab = sp.read.parquet('lake/tm_tables').toPandas().merge(club_x[['club_id', 'comp', 'squad']], on='club_id'); mt = pd.read_parquet('lake/midsem/league_table.parquet')[['comp', 'squad', 'yr', 'position']]
    tab = tab.rename(columns={'season': 'yr'}).merge(mt, on=['comp', 'squad', 'yr'], how='left'); tab['table_pos'] = tab['position'].fillna(tab['tm_pos']).astype(int)
    FE = FE.merge(tab[['comp', 'squad', 'yr', 'table_pos']], on=['comp', 'squad', 'yr'], how='left'); assert FE['table_pos'].notna().all()
    giants = {c for v in MID['GIANTS'].values() for c in v}; FE['is_giant'] = FE['squad'].isin(giants); FE['elite_club'] = FE['is_giant'] | (FE['table_pos'] <= MID['ELITE_TOP_N'])
    FE['feeder'] = ~FE['elite_club'] & (FE['age_num'] <= MID['FEEDER_MAX_AGE']) & (FE['min_num'] >= MID['FEEDER_MIN_MIN']); FE['elite'] = (FE['table_pos'] <= MID['ELITE_TOP_N']) & (FE['min_num'] >= MID['ELITE_MIN_MIN'])
    FE['x_starter_share'] = FE['min_num'] / (FE['matches'] * 90.0)
    mid = pd.read_parquet('lake/midsem/lake.parquet', columns=['player', 'squad', 'yr', 'best_tier', 'labeled'] + ST.AF + ST.RAW).rename(columns={c: c + '_mid' for c in ST.AF + ST.RAW})
    FE = FE.merge(mid, on=['player', 'squad', 'yr'], how='left'); FE['labeled'] = FE['labeled'].fillna(False).astype(bool)
    ctx.to_parquet('lake/team_style.parquet', index=False); ELW.to_parquet('lake/elas_wf.parquet', index=False); OFW.to_parquet('lake/offsets_wf.parquet', index=False)
    both = FE.dropna(subset=[ST.RAW[0] + '_mid']); raw_same = float(np.mean([np.allclose(both[r], both[r + '_mid'], atol=2e-3) for r in ST.RAW])); neu_corr = float(np.mean([np.corrcoef(both[f], both[f + '_mid'])[0, 1] for f in ST.AF]))
    rows['fbref'] = write(sp.createDataFrame(FE.drop(columns=[c + '_mid' for c in ST.AF + ST.RAW])).withColumn('season', F.col('yr')).withColumn('league', F.col('comp')), 'fbref')
sup = ELW[ELW['term'] != 'x_str'].groupby('cutoff')['significant'].sum().to_dict()
print(f"player-seasons: {len(FE):,} ({(FE['yr'] == 2024).sum():,} from 2024-25); raw per-90 identical to the mid-sem table on {raw_same:.0%} of metrics; walk-forward neutral values correlate {neu_corr:.3f} with the mid-sem all-season fit")
print(f"2023-24 overlap of the two FBref sources ({n_chk} players): lowest per-metric correlation {min(src_agree.values()):.3f}; supported elasticities by cut-off: {sup}")

lake/fbref: 11,792 rows
[walk-forward style model] 3.5 s
player-seasons: 11,792 (1,437 from 2024-25); raw per-90 identical to the mid-sem table on 100% of metrics; walk-forward neutral values correlate 0.996 with the mid-sem all-season fit
2023-24 overlap of the two FBref sources (1600 players): lowest per-metric correlation 1.000; supported elasticities by cut-off: {2017: 31, 2018: 40, 2019: 42, 2020: 42, 2021: 48, 2022: 50, 2023: 47, 2024: 47}


## 4. Checks: match rate per league-season, a random sample to eyeball, and the unmatched feeders

In [5]:
FBm = FB.merge(X[['player', 'squad', 'yr', 'player_id', 'confidence', 'fine_pos']], on=['player', 'squad', 'yr'], how='left'); FBm['matched'] = FBm['player_id'].notna()
rate_all = FBm.pivot_table(index='comp', columns='yr', values='matched', aggfunc='mean').round(3)
rate_900 = FBm[FBm['min_num'] >= 900].pivot_table(index='comp', columns='yr', values='matched', aggfunc='mean').round(3)
print('match rate, all outfield rows'); display(rate_all); print('match rate, rows with >= 900 minutes (the modelling population)'); display(rate_900)
print('30 random matches for a manual check:')
display(X.sample(30, random_state=SEED)[['player', 'tm_name', 'squad', 'yr', 'born', 'birth_year', 'sub_position', 'name_sim', 'confidence']])
P3 = X[X['method'] != 'name']; print(f'pass 3 (no name resemblance) added {len(P3)} rows, {P3["player"].nunique()} players:'); display(P3.drop_duplicates('player')[['player', 'tm_name', 'squad', 'yr', 'born', 'min_num', 'tm_minutes']].head(40))
print('20 lowest-confidence matches:'); display(X.nsmallest(20, 'confidence')[['player', 'tm_name', 'squad', 'yr', 'born', 'birth_year', 'sub_position', 'name_sim', 'confidence']])
L = FE[['player', 'squad', 'comp', 'yr', 'feeder', 'age_num', 'min_num']]
Lf = L[L['feeder']].merge(X[['player', 'squad', 'yr', 'player_id']], on=['player', 'squad', 'yr'], how='left')
unm = Lf[Lf['player_id'].isna()].sort_values(['yr', 'comp', 'squad']); unm.to_csv('outputs/unmatched_feeders.csv', index=False)
print(f'unmatched feeders: {len(unm)} of {len(Lf)} feeder-seasons (full list in outputs/unmatched_feeders.csv)'); display(unm.head(40))
same_id = X.groupby('player_id')['player'].nunique(); same_name = X.groupby('player')['player_id'].nunique()
print(f'Transfermarkt ids carrying more than one FBref spelling: {(same_id > 1).sum()}; FBref names that are really several players: {(same_name > 1).sum()}')

match rate, all outfield rows


yr,2017,2018,2019,2020,2021,2022,2023,2024
comp,,,,,,,,
Bundesliga,1.000,0.995,1.000,0.998,0.946,0.993,1.000,0.998
La Liga,0.998,0.971,0.976,0.980,0.879,0.990,0.983,0.989
Ligue 1,0.994,0.986,0.998,0.998,0.908,0.994,0.998,0.985
Premier League,0.996,0.991,0.969,0.996,0.974,1.000,1.000,0.996
Serie A,0.998,0.986,0.955,0.992,0.923,0.998,0.998,0.995


match rate, rows with >= 900 minutes (the modelling population)


yr,2017,2018,2019,2020,2021,2022,2023,2024
comp,,,,,,,,
Bundesliga,1.000,0.996,1.000,1.000,0.996,0.988,1.000,1.000
La Liga,1.000,0.997,0.990,0.984,0.997,0.994,0.994,0.997
Ligue 1,1.000,1.000,1.000,0.997,0.987,1.000,1.000,1.000
Premier League,0.997,1.000,0.990,0.997,0.994,1.000,1.000,1.000
Serie A,1.000,1.000,0.994,0.994,0.997,1.000,1.000,1.000


30 random matches for a manual check:


,player,tm_name,squad,yr,born,birth_year,sub_position,name_sim,confidence
17012,Nadiem Amiri,Nadiem Amiri,Mainz 05,2024,1996.0,1996.0,Central Midfield,1.000000,1.000
16006,Franck Tabanou,Franck Tabanou,Guingamp,2018,1989.0,1989.0,Left-Back,1.000000,1.000
344,Morgan Sanson,Morgan Sanson,Marseille,2020,1994.0,1994.0,Central Midfield,1.000000,1.000
7671,Xaver Schlager,Xaver Schlager,Wolfsburg,2019,1997.0,1997.0,Central Midfield,1.000000,1.000
2225,Christian Eriksen,Christian Eriksen,Manchester Utd,2023,1992.0,1992.0,Attacking Midfield,1.000000,1.000
3886,Papu Gómez,Papu Gómez,Atalanta,2020,1988.0,1988.0,Left Winger,1.000000,0.900
8674,Kyle Naughton,Kyle Naughton,Swansea City,2017,1988.0,1988.0,Right-Back,1.000000,1.000
4580,Bartosz Bereszyński,Bartosz Bereszyński,Napoli,2022,1992.0,1992.0,Right-Back,1.000000,1.000
2616,Soumaïla Coulibaly,Soumaïla Coulibaly,Brest,2024,2003.0,2003.0,Centre-Back,1.000000,1.000
4391,Hugo Duro,Hugo Duro,Getafe,2018,1999.0,1999.0,Centre-Forward,1.000000,0.900


... (3 outputs left out)

,player,tm_name,squad,yr,born,birth_year,sub_position,name_sim,confidence
237,José Luis García Vayá,Pepelu,Valencia,2024,1998.0,1998.0,Defensive Midfield,0.142857,0.5
329,Jesús Santiago,Yellu Santiago,Getafe,2023,2004.0,2004.0,Central Midfield,0.142857,0.5
373,Nelson Sissoko,Alpha Sissoko,Saint-Étienne,2020,1997.0,1997.0,Right-Back,0.642857,0.5
551,Kasim Nuhu,Kasim Adams,Hoffenheim,2020,1995.0,1995.0,Centre-Back,0.181818,0.5
593,Simeon Nwankwo,Simy,Salernitana,2023,1992.0,1992.0,Centre-Forward,0.214286,0.5
1139,Ndombe Mubele,Firmin Mubele,Toulouse,2018,1994.0,1994.0,Centre-Forward,0.153846,0.5
1179,Simeon Nwankwo,Simy,Crotone,2020,1992.0,1992.0,Centre-Forward,0.214286,0.5
1298,Souleymane Touré,Isaak Touré,Udinese,2024,2003.0,2003.0,Centre-Back,0.437500,0.5
1461,Guy Ngosso,Guy N'Gosso,Amiens,2017,1985.0,1985.0,Defensive Midfield,0.181818,0.5
1582,Musa Al-Taamari,Mousa Tamari,Montpellier,2024,1997.0,1997.0,Right Winger,0.666667,0.5


unmatched feeders: 13 of 3154 feeder-seasons (full list in outputs/unmatched_feeders.csv)


,player,squad,comp,yr,feeder,age_num,min_num,player_id
1009,Alejandro Pozo Pozo,Mallorca,La Liga,2019,True,20.0,1498,NaN
998,Samir Santos,Udinese,Serie A,2019,True,24.0,1546,NaN
1266,Alejandro Pozo Pozo,Eibar,La Liga,2020,True,21.0,2190,NaN
1388,Yunus Musah,Valencia,La Liga,2020,True,17.0,1530,NaN
1533,Vagner Gonçalves,Metz,Ligue 1,2020,True,24.0,967,NaN
1915,Nico Ribaudo,Espanyol,La Liga,2021,True,20.0,1116,NaN
1582,Conor Gallagher,Crystal Palace,Premier League,2021,True,21.0,2845,NaN
1728,Armando Broja,Southampton,Premier League,2021,True,19.0,1975,NaN
2242,Agustín Rogel,Hertha BSC,Bundesliga,2022,True,24.0,1426,NaN
2346,Márton Dárdai,Hertha BSC,Bundesliga,2022,True,20.0,947,NaN


Transfermarkt ids carrying more than one FBref spelling: 13; FBref names that are really several players: 39


In [6]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.6, 2.6)); im = ax.imshow(rate_900.values, vmin=0.9, vmax=1, cmap=SEQ); ax.set_xticks(range(rate_900.shape[1]), rate_900.columns); ax.set_yticks(range(len(rate_900)), rate_900.index)
for i in range(rate_900.shape[0]):
    for j in range(rate_900.shape[1]): ax.text(j, i, f'{rate_900.values[i, j]:.0%}', ha='center', va='center', fontsize=8, color='white' if rate_900.values[i, j] > .96 else 'black')
ax.set_title('FBref to Transfermarkt match rate (players with >= 900 min)'); fig.colorbar(im, shrink=.8); os.makedirs('outputs/figs', exist_ok=True); fig.savefig('outputs/figs/p1_match_rate.png', bbox_inches='tight'); plt.show()
_ = put_metrics('phase1', {'seasons': '2017-18 to 2024-25', 'player_seasons': len(FE), 'player_seasons_2024': int((FE['yr'] == 2024).sum()), 'source_overlap_players': n_chk, 'source_overlap_min_corr': min(src_agree.values()), 'raw_identical_share': raw_same, 'wf_neutral_corr_with_midsem': neu_corr, 'supported_elasticities_by_cutoff': {int(k): int(v) for k, v in sup.items()}, 'lake_rows': rows, 'runtime_s': RT, 'clubs_mapped': len(club_x), 'club_vote_share_min': float(club_x['share'].min()), 'rows_linked': len(X), 'rows_total': len(FB),
                       'match_rate_all': float(FBm['matched'].mean()), 'match_rate_900': float(FBm.loc[FBm['min_num'] >= 900, 'matched'].mean()), 'match_rate_900_min_cell': float(np.nanmin(rate_900.values)),
                       'pass3_rows': len(P3), 'pass3_players': int(P3['player'].nunique()), 'feeders_total': len(Lf), 'feeders_unmatched': len(unm), 'mean_confidence': float(X['confidence'].mean()), 'ids_multi_spelling': int((same_id > 1).sum()), 'names_multi_id': int((same_name > 1).sum())})